### Instalando bibliotecas

In [12]:
%pip install pandas scikit-learn matplotlib imbalanced-learn

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.3 -> 26.3
[notice] To update, run: python.exe -m pip install --upgrade pip


In [13]:
import pandas as pd

df = pd.read_csv('loan.csv')

print(df.shape)

print(df['Loan_Status'].value_counts())

df = df.drop(columns=[
    'Loan_ID',
    'CoapplicantIncome',
    'Loan_Amount_Term',
    'Credit_History',
    'Property_Area'
])


df.isnull().sum()

(614, 13)
Loan_Status
Y    422
N    192
Name: count, dtype: int64


Gender             13
Married             3
Dependents         15
Education           0
Self_Employed      32
ApplicantIncome     0
LoanAmount         22
Loan_Status         0
dtype: int64

In [14]:
# Ver linhas com dados faltantes
df.isnull().sum()

# Preencher dados faltantes com a Moda

from sklearn.impute import SimpleImputer

colunas_moda = [
    'Dependents',
    'Self_Employed',
    'Married',
    'Gender'
]

# Para features categóricas
imputerC = SimpleImputer(strategy='most_frequent')

# Para features númericas

imputerN = SimpleImputer(strategy='median')

df[['LoanAmount']] = imputerN.fit_transform(df[['LoanAmount']])

df[colunas_moda] = imputerC.fit_transform(df[colunas_moda])

# Confirmar se não há dados faltantes (item d do enunciado)
print("Dados faltantes após imputação:")
print(df.isnull().sum())


Dados faltantes após imputação:
Gender             0
Married            0
Dependents         0
Education          0
Self_Employed      0
ApplicantIncome    0
LoanAmount         0
Loan_Status        0
dtype: int64


In [15]:
# Transformando features categóricas com LabelEncoder

from sklearn.preprocessing import LabelEncoder

colunas_encoder = [
    'Gender',
    'Married',
    'Education',
    'Self_Employed',
    'Loan_Status'
]

label_encoders = {}

for coluna in colunas_encoder:
    le = LabelEncoder()
    df[coluna] = le.fit_transform(df[coluna])
    label_encoders[coluna] = le


# A feature Dependents sendo reformulada entre 0, 1, 2 e 3 apenas
df['Dependents'] = df['Dependents'].replace('3+', '3')

df['Dependents'] = df['Dependents'].astype(int)



In [16]:
X = df.drop('Loan_Status', axis=1)
y = df['Loan_Status']

from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=40, stratify=y)

X_train.shape , y_train.shape, X_test.shape, y_test.shape

print(X_train.head())

     Gender  Married  Dependents  Education  Self_Employed  ApplicantIncome  \
101       1        0           0          0              0             4843   
116       0        1           0          0              0             3167   
191       1        0           0          0              0            12000   
237       0        0           0          0              1             3463   
222       1        0           0          0              0             2971   

     LoanAmount  
101       151.0  
116       154.0  
191       164.0  
237       122.0  
222       144.0  


In [17]:
from imblearn.pipeline import Pipeline
from imblearn.over_sampling import SMOTE
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.linear_model import SGDClassifier

modelos = {
    'Regressao Logistica': {
        'modelo': LogisticRegression(max_iter=1000, random_state=42),
        'parametros': {
            'modelo__C': [0.1, 1, 10]
        }
    },

    'Arvore de Decisao': {
        'modelo': DecisionTreeClassifier(random_state=42),
        'parametros': {
            'modelo__max_depth': [3, 5, 10, None],
            'modelo__min_samples_split': [2, 5]
        }
    },

    'Random Forest': {
        'modelo': RandomForestClassifier(random_state=42),
        'parametros': {
            'modelo__n_estimators': [100, 200],
            'modelo__max_depth': [5, 10, None]
        }
    },

    'KNN': {
        'modelo': KNeighborsClassifier(),
        'parametros': {
            'modelo__n_neighbors': [3, 5, 7],
            'modelo__weights': ['uniform', 'distance']
        }
    },

    'SGD': {
    'modelo': SGDClassifier(
        random_state=42,
        max_iter=1000,
        tol=1e-3
    ),
    'parametros': {
        'modelo__loss': ['hinge', 'log_loss'],
        'modelo__alpha': [0.0001, 0.001, 0.01],
        'modelo__penalty': ['l2', 'l1']
    }
}
}


In [18]:
# Pipeline com SMOTE

from sklearn.compose import ColumnTransformer

colunas_numericas = [
    'Dependents',
    'ApplicantIncome',
    'LoanAmount'
]

preprocessador = ColumnTransformer(
    transformers=[
        ('numericas', StandardScaler(), colunas_numericas)
    ],
    remainder='passthrough'
)

# Grid Search e Validação Cruzada

from sklearn.model_selection import GridSearchCV, StratifiedKFold

resultados = {}

cv = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

for nome, configuracao in modelos.items():

    pipeline = Pipeline([
        ('preprocessamento', preprocessador),
        ('smote', SMOTE(random_state=42)),
        ('modelo', configuracao['modelo'])
    ])

    grid = GridSearchCV(
        estimator=pipeline,
        param_grid=configuracao['parametros'],
        cv=cv,
        scoring='roc_auc',
        n_jobs=-1,
        verbose=1
    )

    grid.fit(X_train, y_train)

    resultados[nome] = grid

    print(f'\nModelo: {nome}')
    print('Melhores parâmetros:', grid.best_params_)
    print('Melhor AUC na validação cruzada:', grid.best_score_)

Fitting 5 folds for each of 3 candidates, totalling 15 fits

Modelo: Regressao Logistica
Melhores parâmetros: {'modelo__C': 1}
Melhor AUC na validação cruzada: 0.5630110359020836
Fitting 5 folds for each of 8 candidates, totalling 40 fits

Modelo: Arvore de Decisao
Melhores parâmetros: {'modelo__max_depth': None, 'modelo__min_samples_split': 2}
Melhor AUC na validação cruzada: 0.5397568608570052
Fitting 5 folds for each of 6 candidates, totalling 30 fits

Modelo: Random Forest
Melhores parâmetros: {'modelo__max_depth': None, 'modelo__n_estimators': 200}
Melhor AUC na validação cruzada: 0.5429855230488922
Fitting 5 folds for each of 6 candidates, totalling 30 fits

Modelo: KNN
Melhores parâmetros: {'modelo__n_neighbors': 7, 'modelo__weights': 'uniform'}
Melhor AUC na validação cruzada: 0.5167904311459779
Fitting 5 folds for each of 12 candidates, totalling 60 fits

Modelo: SGD
Melhores parâmetros: {'modelo__alpha': 0.001, 'modelo__loss': 'log_loss', 'modelo__penalty': 'l1'}
Melhor AUC n

In [19]:
# Métrica AUC-ROC

from sklearn.metrics import classification_report, roc_auc_score

for nome, grid in resultados.items():

    melhor_modelo = grid.best_estimator_

    y_pred = melhor_modelo.predict(X_test)

    if hasattr(melhor_modelo, "predict_proba"):
        y_prob = melhor_modelo.predict_proba(X_test)[:, 1]
    else:
        y_prob = melhor_modelo.decision_function(X_test)

    auc = roc_auc_score(y_test, y_prob)

    print('\n' + '=' * 50)
    print(f'MODELO: {nome}')
    print('=' * 50)

    print('\nClassification Report:')
    print(classification_report(y_test, y_pred))

    print(f'AUC-ROC: {auc:.4f}')


MODELO: Regressao Logistica

Classification Report:
              precision    recall  f1-score   support

           0       0.36      0.58      0.44        38
           1       0.74      0.54      0.63        85

    accuracy                           0.55       123
   macro avg       0.55      0.56      0.54       123
weighted avg       0.62      0.55      0.57       123

AUC-ROC: 0.5675

MODELO: Arvore de Decisao

Classification Report:
              precision    recall  f1-score   support

           0       0.24      0.26      0.25        38
           1       0.65      0.62      0.64        85

    accuracy                           0.51       123
   macro avg       0.45      0.44      0.44       123
weighted avg       0.53      0.51      0.52       123

AUC-ROC: 0.4433

MODELO: Random Forest

Classification Report:
              precision    recall  f1-score   support

           0       0.27      0.32      0.29        38
           1       0.67      0.62      0.65        85


In [20]:
# Comparação com os 5 modelos

import pandas as pd

comparacao = []

for nome, grid in resultados.items():

    modelo = grid.best_estimator_

    y_pred = modelo.predict(X_test)
    if hasattr(modelo, "predict_proba"):
        y_prob = modelo.predict_proba(X_test)[:, 1]
    else:
        y_prob = modelo.decision_function(X_test)

    auc = roc_auc_score(y_test, y_prob)

    comparacao.append({
        'Modelo': nome,
        'AUC-ROC': auc,
        'Melhor AUC CV': grid.best_score_,
        'Melhores parâmetros': grid.best_params_
    })

df_comparacao = pd.DataFrame(comparacao)

df_comparacao = df_comparacao.sort_values(
    by='AUC-ROC',
    ascending=False
)

display(df_comparacao)

,Modelo,AUC-ROC,Melhor AUC CV,Melhores parâmetros
0,Regressao Logistica,0.567492,0.563011,{'modelo__C': 1}
4,SGD,0.557121,0.572479,"{'modelo__alpha': 0.001, 'modelo__loss': 'log_..."
3,KNN,0.482972,0.516790,"{'modelo__n_neighbors': 7, 'modelo__weights': ..."
2,Random Forest,0.471517,0.542986,"{'modelo__max_depth': None, 'modelo__n_estimat..."
1,Arvore de Decisao,0.443344,0.539757,"{'modelo__max_depth': None, 'modelo__min_sampl..."


In [21]:
# Melhor modelo

melhor_nome = max(
    resultados,
    key=lambda nome: roc_auc_score(
        y_test,
        resultados[nome].best_estimator_.predict_proba(X_test)[:, 1]
        if hasattr(resultados[nome].best_estimator_, 'predict_proba')
        else resultados[nome].best_estimator_.decision_function(X_test)
    )
)

melhor_modelo = resultados[melhor_nome].best_estimator_

print("Melhor modelo:", melhor_nome)

# Salvar o modelo

import joblib

joblib.dump(melhor_modelo, 'modelo_loan_status.pkl')
joblib.dump(label_encoders, 'encoders.pkl')

Melhor modelo: Regressao Logistica


['encoders.pkl']